# Data Visualization & Storytelling — Breast Cancer Dataset

Good visualizations communicate insight, not just data. This notebook builds a 4-panel analytical dashboard — the kind used in stakeholder presentations — and walks through each chart's design purpose.

**Key libraries:** matplotlib (layout/control), seaborn (statistical charts)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import load_breast_cancer

sns.set_theme(style='whitegrid')
raw = load_breast_cancer(as_frame=True)
df = raw.frame
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})
palette = {'Malignant': '#e74c3c', 'Benign': '#2ecc71'}
print(f"Loaded: {df.shape}")

## Panel A — Class Distribution

The first thing any stakeholder wants to know: how many cases of each class? A bar chart with data labels is the clearest answer. We avoid pie charts here — they make magnitude comparisons harder.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['diagnosis'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[palette[k] for k in counts.index],
              edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Breast Cancer Diagnoses in Dataset', fontsize=13, pad=10)
ax.set_ylabel('Number of Cases')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

## Panel B — Feature Separation (Boxplots)

Do the two classes differ on key measurements? Boxplots show median, spread, and outliers — more informative than bar charts for continuous data. We choose 4 features that showed strong class separation in the EDA.

In [ ]:
features_to_plot = ['mean radius', 'mean texture', 'mean concavity', 'mean symmetry']

fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, col in zip(axes, features_to_plot):
    data_by_class = [df[df['diagnosis'] == d][col].values for d in palette]
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], palette.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(palette.keys(), rotation=15)
    ax.set_title(col.replace('mean ', '').title(), fontsize=10)

fig.suptitle('Feature Distributions by Diagnosis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Panel C — Two-Feature Scatter Plot

Scatter plots reveal whether two features *together* separate the classes. This is a useful visual check before building a classification model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, color in palette.items():
    subset = df[df['diagnosis'] == label]
    ax.scatter(subset['mean radius'], subset['mean concavity'],
               color=color, label=label, alpha=0.6,
               edgecolors='white', linewidths=0.5, s=50)
ax.set_xlabel('Mean Radius', fontsize=11)
ax.set_ylabel('Mean Concavity', fontsize=11)
ax.set_title('Mean Radius vs Concavity by Diagnosis', fontsize=13)
ax.legend(title='Diagnosis', fontsize=10)
plt.tight_layout()
plt.show()

## Panel D — Full Dashboard

Combining all panels into one figure creates a report-ready analytical dashboard.

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# A: class distribution
counts = df['diagnosis'].value_counts()
bars = ax1.bar(counts.index, counts.values,
               color=[palette[k] for k in counts.index],
               edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             str(count), ha='center', fontweight='bold')
ax1.set_title('A. Class Distribution', fontsize=12, loc='left')
ax1.set_ylim(0, counts.max() * 1.15)

# B: boxplot
for i, (label, color) in enumerate(palette.items()):
    bp = ax2.boxplot(df[df['diagnosis'] == label]['mean radius'].values,
                     positions=[i], patch_artist=True, widths=0.5,
                     medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.7)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(palette.keys())
ax2.set_title('B. Mean Radius by Diagnosis', fontsize=12, loc='left')
ax2.set_ylabel('Mean Radius')

# C: scatter
for label, color in palette.items():
    s = df[df['diagnosis'] == label]
    ax3.scatter(s['mean radius'], s['mean concavity'],
                color=color, label=label, alpha=0.6, edgecolors='white', s=40)
ax3.set_xlabel('Mean Radius')
ax3.set_ylabel('Mean Concavity')
ax3.set_title('C. Radius vs Concavity', fontsize=12, loc='left')
ax3.legend(fontsize=9)

# D: violin
violin_data = [df[df['diagnosis'] == d]['mean texture'].values for d in palette]
parts = ax4.violinplot(violin_data, positions=[0, 1], showmedians=True)
for pc, color in zip(parts['bodies'], palette.values()):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
ax4.set_xticks([0, 1])
ax4.set_xticklabels(palette.keys())
ax4.set_title('D. Texture Distribution (Violin)', fontsize=12, loc='left')
ax4.set_ylabel('Mean Texture')

fig.suptitle('Breast Cancer Dataset — Analytical Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Visualization Best Practices

1. **Label everything** — axes, titles, legends, and key data points.
2. **Use color with intent** — consistent class colors across all panels.
3. **Match chart type to data type** — bars for counts, box/violin for distributions, scatter for relationships.
4. **Remove chart junk** — avoid grid lines and borders that don't add information.
5. **Tell a story** — panels A-D build toward a conclusion: the classes are clearly separable on size and shape features.